<a href="https://colab.research.google.com/github/Pinizov/EasyPay-Autonomous-Kiosk/blob/main/quickstarts/Get_started_LiveAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
import kagglehub
path = kagglehub.dataset_download("aliiihussain/amazon-sales-dataset")

Using Colab cache for faster access to the 'amazon-sales-dataset' dataset.


In [50]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [53]:
import google.generativeai as genai
from google.colab import userdata
import json

# Ensure the API key is configured
# This assumes GOOGLE_API_KEY is already set in Colab secrets and retrieved by the previous cell.
# If not, please run the cell for GOOGLE_API_KEY setup.

# Fallback for API_KEY if not already set (though it should be by earlier cells)
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
except: # Catching an error when user has not added API key in the secrets.
    print("Warning: GOOGLE_API_KEY not found in Colab secrets. Please set it to use the Gemini API.")

# Define the generative model
# Using the previously defined MODEL variable for consistency
model = genai.GenerativeModel(MODEL)

def extract_categories_tags_with_llm(description):
    if not isinstance(description, str) or not description:
        return {'categories': [], 'tags': []}

    prompt = f"""Extract categories and tags from the following product description.
Return the output as a JSON object with two keys: 'categories' (a list of strings) and 'tags' (a list of strings).
If no categories or tags are found, return empty lists.

Product Description: {description}

Example Output:
{{"categories": ["Motorcycle Parts"], "tags": ["KTM", "exhaust", "protection"]}}"""

    try:
        response = model.generate_content(prompt)
        # Assuming the model returns a JSON string, try to parse it
        # Sometimes LLMs add markdown formatting, so try to clean it first
        cleaned_response = response.text.replace('```json', '').replace('```', '').strip()
        data = json.loads(cleaned_response)
        categories = data.get('categories', [])
        tags = data.get('tags', [])
        return {'categories': categories, 'tags': tags}
    except Exception as e:
        print(f"Error processing description '{description[:50]}...': {e}")
        return {'categories': [], 'tags': []}

# Apply the function to the 'Original Description' column
print("Extracting categories and tags using LLM. This may take a while...")
df_orders['LLM_Extracted_Data'] = df_orders['Original Description'].apply(extract_categories_tags_with_llm)

# Expand the dictionary column into separate 'Categories' and 'Tags' columns
df_orders['Categories'] = df_orders['LLM_Extracted_Data'].apply(lambda x: x['categories'])
df_orders['Tags'] = df_orders['LLM_Extracted_Data'].apply(lambda x: x['tags'])

# Drop the intermediate column if no longer needed
df_orders = df_orders.drop(columns=['LLM_Extracted_Data'])

print("Extraction complete. Displaying first 5 rows with new columns:")
display(df_orders[['Original Description', 'Categories', 'Tags']].head())

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Extracting categories and tags using LLM. This may take a while...
Error processing description '22mm 7/8-inch Left Aluminum Alloy Clutch Lever Han...': 
  No API_KEY or ADC found. Please either:
    - Set the `GOOGLE_API_KEY` environment variable.
    - Manually pass the key with `genai.configure(api_key=my_api_key)`.
    - Or set up Application Default Credentials, see https://ai.google.dev/gemini-api/docs/oauth for more information.
Error processing description 'Motorcycle KOSO Water Temperature Mini Meter For X...': 
  No API_KEY or ADC found. Please either:
    - Set the `GOOGLE_API_KEY` environment variable.
    - Manually pass the key with `genai.configure(api_key=my_api_key)`.
    - Or set up Application Default Credentials, see https://ai.google.dev/gemini-api/docs/oauth for more information.
Error processing description '2024 For KTM EXC 250 300 EXC250 EXC300 XC XCW TPI ...': 
  No API_KEY or ADC found. Please either:
    - Set the `GOOGLE_API_KEY` environment variable.
    -

,Original Description,Categories,Tags
0,,[],[]
1,22mm 7/8-inch Left Aluminum Alloy Clutch Lever...,[],[]
2,Motorcycle KOSO Water Temperature Mini Meter F...,[],[]
3,2024 For KTM EXC 250 300 EXC250 EXC300 XC XCW ...,[],[]
4,Motorcycle License Plate Mount Bracket Tail Re...,[],[]


In [51]:
import re

# Function to clean product names by removing URLs and extra spaces
def clean_product_name(text):
    if isinstance(text, str):
        # Remove URLs (http/https) from the text
        cleaned_text = re.sub(r'https?://[\w./\-]+(?:\?[\w=&.\-]+)*', '', text)
        # Remove common AliExpress artifacts that appear in product names after URLs or prices
        cleaned_text = re.sub(r'Total:\tAdd to cart\tRemove', '', cleaned_text)
        # Remove extra tabs and spaces
        cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
        return cleaned_text
    return text

# Apply the cleaning function to the 'Product Name' column
df_orders['Cleaned Product Name'] = df_orders['Product Name'].apply(clean_product_name)

# Create 'Original Description' from the cleaned product name
df_orders['Original Description'] = df_orders['Cleaned Product Name']

# Create 'Shortened Description' by taking the first 15 words
def shorten_description(text, num_words=15):
    if isinstance(text, str):
        words = text.split()
        return ' '.join(words[:num_words]) + ('...' if len(words) > num_words else '')
    return text

df_orders['Shortened Description'] = df_orders['Original Description'].apply(shorten_description)

display(df_orders[['Product Name', 'Cleaned Product Name', 'Original Description', 'Shortened Description']].head())

,Product Name,Cleaned Product Name,Original Description,Shortened Description
0,https://www.aliexpress.com/item/10050060877736...,,,
1,https://www.aliexpress.com/item/10050032000382...,22mm 7/8-inch Left Aluminum Alloy Clutch Lever...,22mm 7/8-inch Left Aluminum Alloy Clutch Lever...,22mm 7/8-inch Left Aluminum Alloy Clutch Lever...
2,https://www.aliexpress.com/item/32907366553.ht...,Motorcycle KOSO Water Temperature Mini Meter F...,Motorcycle KOSO Water Temperature Mini Meter F...,Motorcycle KOSO Water Temperature Mini Meter F...
3,https://www.aliexpress.com/item/10050071709500...,2024 For KTM EXC 250 300 EXC250 EXC300 XC XCW ...,2024 For KTM EXC 250 300 EXC250 EXC300 XC XCW ...,2024 For KTM EXC 250 300 EXC250 EXC300 XC XCW ...
4,https://www.aliexpress.com/item/10050073552758...,Motorcycle License Plate Mount Bracket Tail Re...,Motorcycle License Plate Mount Bracket Tail Re...,Motorcycle License Plate Mount Bracket Tail Re...


Regarding image links:

Previously, we attempted to web scrape image URLs from AliExpress product pages, but this proved difficult because AliExpress uses dynamic content loaded via JavaScript, which `requests` and `BeautifulSoup` cannot easily parse. We also discussed the possibility of using an official AliExpress API, but that would require you to manually find a valid API endpoint through browser developer tools, which is outside of my current capabilities.

Without a direct way to programmatically obtain image URLs for these products, I cannot provide downloadable links. If you can manually extract image URLs or provide a new method to access them, I can then help with a placeholder for AI image searching or downloading.

For example, if you were able to gather a list of image URLs, you could use a function like the one below for further processing or AI analysis:

In [52]:
def search_ai_for_image(image_url):
    # This is a placeholder function.
    # In a real scenario, you would integrate with an AI image recognition service,
    # like Google Cloud Vision API or a custom ML model, to analyze the image.
    # It might return tags, descriptions, categories, or perform reverse image search.
    print(f"Performing AI search for image: {image_url}")
    # Example: return a dummy result
    return {
        'ai_description': f'AI analysis of image from {image_url[:50]}...',
        'ai_tags': ['motorcycle part', 'accessory']
    }

# Example of how you would use it if you had image URLs in your DataFrame:
# df_orders['Image AI Search Result'] = df_orders['Image URL'].apply(search_ai_for_image)

# Since we don't have 'Image URL' yet, this cell is just for demonstration of the concept.